# Reintegration Readiness Prediction
Predicting which residents are ready for reintegration into the community.

## 1. Problem Framing
Reintegration readiness determines whether a shelter resident has achieved the stability, skills, and support network required to successfully transition to independent living. Accurate prediction enables case managers to allocate resources efficiently and time transition plans appropriately.

**Target variable:** `ready_for_reintegration` — binary (1 = assessed as ready within 30 days, 0 = not yet ready).

**Success metric:** Recall ≥ 0.80 for class 1 (do not miss residents who are ready), ROC-AUC ≥ 0.78.

**Stakeholders:** Case management and program staff.

## 2. Data Acquisition & Preparation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

np.random.seed(42)
n = 600

programs = ['Job Training', 'Life Skills', 'Counseling', 'Education', 'None']

df = pd.DataFrame({
    'resident_id': range(1, n + 1),
    'length_of_stay_days': np.random.randint(30, 730, size=n),
    'employment_status': np.random.choice(['Employed', 'Part-time', 'Unemployed'], size=n, p=[0.3, 0.3, 0.4]),
    'housing_plan_in_place': np.random.randint(0, 2, size=n),
    'support_network_score': np.random.randint(0, 11, size=n),
    'mental_health_score': np.random.randint(1, 11, size=n),  # higher = better
    'substance_use_flag': np.random.randint(0, 2, size=n),
    'program_completion_pct': np.random.uniform(0, 1, size=n),
    'case_manager_rating': np.random.randint(1, 6, size=n),
    'prior_reintegration_attempts': np.random.randint(0, 4, size=n),
    'primary_program': np.random.choice(programs, size=n),
})

# Encode categorical
le_emp = LabelEncoder()
df['employment_encoded'] = le_emp.fit_transform(df['employment_status'])
df = pd.get_dummies(df, columns=['primary_program'], drop_first=True)

# Simulate label
ready_prob = (
    0.1
    + 0.25 * df['housing_plan_in_place']
    + 0.02 * df['support_network_score']
    + 0.02 * df['mental_health_score']
    + 0.15 * df['program_completion_pct']
    + 0.03 * df['case_manager_rating']
    - 0.15 * df['substance_use_flag']
    - 0.05 * df['prior_reintegration_attempts']
).clip(0.05, 0.95)
df['ready_for_reintegration'] = np.random.binomial(1, ready_prob)

print(df.shape)
print(df['ready_for_reintegration'].value_counts())
df.head()

In [ ]:
feature_cols = [
    'length_of_stay_days', 'employment_encoded', 'housing_plan_in_place',
    'support_network_score', 'mental_health_score', 'substance_use_flag',
    'program_completion_pct', 'case_manager_rating', 'prior_reintegration_attempts'
] + [c for c in df.columns if c.startswith('primary_program_')]

X = df[feature_cols]
y = df['ready_for_reintegration']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Exploration

In [ ]:
import matplotlib.pyplot as plt

# Readiness rate by employment status
emp_ready = df.groupby('employment_status')['ready_for_reintegration'].mean().sort_values()
emp_ready.plot(kind='barh', color='steelblue', figsize=(7, 3))
plt.title('Readiness Rate by Employment Status')
plt.xlabel('Proportion Ready')
plt.tight_layout()
plt.show()

# Correlation heatmap (numeric features)
num_cols = ['length_of_stay_days', 'support_network_score', 'mental_health_score',
            'program_completion_pct', 'case_manager_rating', 'ready_for_reintegration']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(num_cols, fontsize=8)
plt.colorbar(im, ax=ax)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Modeling
### 4a. Predictive Model — Gradient Boosting Classifier
### 4b. Causal / Explanatory Model — Logistic Regression with statsmodels (odds ratios)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# --- 4a: Predictive model ---
gbc = GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
gbc.fit(X_train, y_train)

y_pred_proba = gbc.predict_proba(X_test)[:, 1]
y_pred = gbc.predict(X_test)
print("Gradient Boosting training complete.")

In [ ]:
# --- 4b: Causal / Explanatory model — Logistic Regression via statsmodels ---
try:
    import statsmodels.api as sm

    # Use scaled features for stable estimation
    X_train_sm = sm.add_constant(pd.DataFrame(X_train_sc, columns=feature_cols))
    logit_model = sm.Logit(y_train.reset_index(drop=True), X_train_sm)
    result = logit_model.fit(disp=False)
    print(result.summary2())

    # Odds ratios with 95% CI
    odds = pd.DataFrame({
        'OR': np.exp(result.params),
        'CI_lower': np.exp(result.conf_int()[0]),
        'CI_upper': np.exp(result.conf_int()[1]),
        'p_value': result.pvalues
    }).drop('const', errors='ignore').sort_values('OR', ascending=False)

    print("\nOdds Ratios (top 10):")
    print(odds.head(10).round(3))
except ImportError:
    print("statsmodels not installed — skipping. Install with: pip install statsmodels")

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay
)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Not Ready', 'Ready']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='GBC')
axes[0].set_title('ROC Curve — Reintegration Readiness')
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Not Ready', 'Ready'], ax=axes[1])
axes[1].set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Feature Selection

In [ ]:
importances = pd.Series(gbc.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
importances.head(12).plot(kind='bar', color='teal')
plt.title('GBC Feature Importances — Reintegration Readiness')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

# Select top 8 features
selected_features = importances.head(8).index.tolist()
print("Selected features:", selected_features)

## 7. Deployment

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

gbc_final = GradientBoostingClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
)
gbc_final.fit(X_train_sel, y_train)

with open('models/reintegration_readiness_model.pkl', 'wb') as f:
    pickle.dump({'model': gbc_final, 'features': selected_features}, f)

print("Model saved to models/reintegration_readiness_model.pkl")

# Score a small batch
sample = X_test_sel.iloc[:5].copy()
sample['readiness_probability'] = gbc_final.predict_proba(sample)[:, 1]
sample['recommendation'] = sample['readiness_probability'].apply(
    lambda p: 'Initiate transition plan' if p >= 0.6 else 'Continue current program'
)
print(sample[['readiness_probability', 'recommendation']])